# Attention U-Net Segmentation Benchmark

Train **Attention U-Net** using the same data split and hyperparameters used for U-Net/ResUNet, then save metrics so all three models can be compared in `main_pipeline.ipynb`.

## Goals
- Train/evaluate Attention U-Net only
- Track Dice, IoU, and HD95
- Save `attention_unet_metrics.json`
- Build/update 3-model comparison CSV (`U-Net`, `ResUNet`, `Attention U-Net`)
- Plot metric comparison using saved model metrics


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# Colab + project setup
print('Setup version: no-auto-mount-v2')
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = importlib.util.find_spec('google.colab') is not None

# Colab occasionally returns a transient 404 during auth propagation.
# To prevent hard crashes, auto-mount is disabled by default.
TRY_MOUNT_DRIVE = False

if IS_COLAB:
    drive_ready = Path('/content/drive/MyDrive').exists()
    if drive_ready:
        print('Google Drive already available at /content/drive')
    elif TRY_MOUNT_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            print('Google Drive mounted at /content/drive')
        except Exception as e:
            print('Warning: Google Drive mount failed (continuing without crash).')
            print('Error:', repr(e))
            print('You can retry manually:')
            print("  from google.colab import drive; drive.mount('/content/drive', force_remount=True)")
    else:
        print('Drive is not mounted. Set TRY_MOUNT_DRIVE=True to auto-mount, or mount manually.')
        print("Manual command: from google.colab import drive; drive.mount('/content/drive', force_remount=True)")
else:
    print('Not running in Colab; skipping Google Drive mount.')

project_candidates = [
    Path('/content/drive/MyDrive/comp4471Project'),
    Path('/content/drive/MyDrive/Comp4471Project'),
    Path('/content/drive/MyDrive/COMP4471Project'),
    Path('/content/comp4471Project'),
    Path('/content/project'),
    Path('/Users/chyo/My Drive/comp4471Project'),
    Path.cwd(),
]

PROJECT_ROOT = next((p for p in project_candidates if (p / 'src' / 'preprocessing.py').exists()), None)
if PROJECT_ROOT is None and Path('/content/drive/MyDrive').exists():
    for p in Path('/content/drive/MyDrive').glob('*'):
        if (p / 'src' / 'preprocessing.py').exists():
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find project root containing src/preprocessing.py. '
        'If Drive mount failed, retry mount or copy project to /content/comp4471Project.'
    )

PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Added project root to sys.path and set working directory.')


def _missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def _pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


requirements = [
    ('medpy', 'medpy'),
    ('albumentations', 'albumentations'),
    ('cv2', 'opencv-python-headless'),
]
packages_to_install = [pkg for module, pkg in requirements if _missing(module)]

if packages_to_install:
    print('Installing:', ', '.join(packages_to_install))
    _pip_install(packages_to_install)
    print('Install complete.')
else:
    print('All required libraries are already installed.')


In [ ]:
import importlib
import inspect
import json
import random
import time
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import src.preprocessing as _preprocessing
import src.segmentation as _segmentation

# Reload project modules so Colab picks up latest Drive edits.
importlib.reload(_preprocessing)
importlib.reload(_segmentation)

from src.preprocessing import discover_image_mask_pairs, stratified_split_pairs, SegmentationDataset
from src.segmentation import compute_segmentation_metrics, UNet, ResUNet
from src.utils import AverageMeter, EarlyStopping, fmt_time


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
# Shared hyperparameters (match U-Net / ResUNet notebooks).
DATA_DIR = None
SPLIT_RATIOS = (0.8, 0.1, 0.1)

IMG_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 130
LR = 3e-4
PATIENCE = 12
BASE_FILTERS = 64
WEIGHT_DECAY = 1e-5
THRESHOLD = 0.5
FORCE_RETRAIN = False

BASE_OUTPUT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()
MODEL_DIR = BASE_OUTPUT_ROOT / 'models' / 'segmentation'
METRICS_DIR = BASE_OUTPUT_ROOT / 'outputs' / 'metrics'
FIGURES_DIR = BASE_OUTPUT_ROOT / 'outputs' / 'figures'
for p in (MODEL_DIR, METRICS_DIR, FIGURES_DIR):
    p.mkdir(parents=True, exist_ok=True)

ATTN_RESULTS_PATH = METRICS_DIR / 'attention_unet_metrics.json'
COMPARISON_CSV_PATH = METRICS_DIR / 'unet_resunet_attention_comparison.csv'
COMPARISON_JSON_PATH = METRICS_DIR / 'unet_resunet_attention_results.json'
METRIC_PLOT_PATH = FIGURES_DIR / 'unet_resunet_attention_metrics.png'
CURVE_PLOT_PATH = FIGURES_DIR / 'attention_unet_training_curves.png'

print('Shared hyperparameters:')
print(f'  IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LR}, PATIENCE={PATIENCE}')


In [ ]:
# Data discovery + split + dataloaders
def _iter_dirs_bfs(base: Path, max_depth: int = 2):
    if not base.exists():
        return
    q = deque([(base, 0)])
    seen = set()
    while q:
        cur, depth = q.popleft()
        try:
            key = str(cur.resolve())
        except Exception:
            key = str(cur)
        if key in seen:
            continue
        seen.add(key)
        yield cur
        if depth >= max_depth:
            continue
        try:
            children = [d for d in cur.iterdir() if d.is_dir()]
        except Exception:
            continue
        for child in children:
            q.append((child, depth + 1))


def _dedupe_paths(paths):
    out = []
    seen = set()
    for p in paths:
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out


explicit_candidates = [
    Path(DATA_DIR) if DATA_DIR else None,
    PROJECT_ROOT / 'data',
    PROJECT_ROOT / 'dataset',
    Path('/content/drive/MyDrive/comp4471Project/data'),
    Path('/content/drive/MyDrive/comp4471Project/dataset'),
]
candidate_roots = [p for p in explicit_candidates if p is not None and p.exists()]
candidate_roots.extend(list(_iter_dirs_bfs(PROJECT_ROOT, max_depth=2)))

drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    keywords = ('dataset', 'data', 'tumor', 'brain', 'mri', 'kaggle', 'archive', 'comp4471')
    for top in drive_root.iterdir():
        if top.is_dir() and any(k in top.name.lower() for k in keywords):
            candidate_roots.append(top)
            candidate_roots.extend(list(_iter_dirs_bfs(top, max_depth=1)))

candidate_roots = _dedupe_paths(candidate_roots)
if not candidate_roots:
    raise FileNotFoundError('No candidate data roots found. Set DATA_DIR manually.')

pseudo_root = PROJECT_ROOT / 'outputs' / 'pseudo_masks_three_model'
kwargs = {}
sig = inspect.signature(discover_image_mask_pairs)
if 'allow_pseudo_fallback' in sig.parameters:
    kwargs['allow_pseudo_fallback'] = True
if 'pseudo_output_dir' in sig.parameters:
    kwargs['pseudo_output_dir'] = str(pseudo_root)

pairs = None
data_root = None
for cand in candidate_roots:
    try:
        maybe_pairs = discover_image_mask_pairs(str(cand), **kwargs)
        if maybe_pairs:
            pairs = maybe_pairs
            data_root = cand
            break
    except FileNotFoundError:
        continue

if pairs is None:
    checked_preview = '\n'.join(f'  - {p}' for p in candidate_roots[:25])
    raise FileNotFoundError(
        'Could not locate a usable segmentation/classification dataset root automatically.\n'
        'Set DATA_DIR to your actual dataset directory and rerun.\n'
        f'Checked roots (first 25):\n{checked_preview}'
    )

print('Selected data root:', data_root)
print('Total image-mask pairs:', len(pairs))

train_pairs, val_pairs, test_pairs = stratified_split_pairs(pairs, ratios=SPLIT_RATIOS, seed=SEED)
print(f'Splits -> Train={len(train_pairs)} Val={len(val_pairs)} Test={len(test_pairs)}')

train_ds = SegmentationDataset(train_pairs, img_size=IMG_SIZE, augment=True)
val_ds = SegmentationDataset(val_pairs, img_size=IMG_SIZE, augment=False)
test_ds = SegmentationDataset(test_pairs, img_size=IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Batches -> Train={len(train_loader)} Val={len(val_loader)} Test={len(test_loader)}')


In [ ]:
# Model definition: Attention U-Net
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(g_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(inter_ch),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(x_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(inter_ch),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.att = AttentionGate(g_ch=out_ch, x_ch=skip_ch, inter_ch=max(out_ch // 2, 1))
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        skip = self.att(x, skip)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        f = base_filters
        self.enc1 = ConvBlock(in_channels, f)
        self.enc2 = ConvBlock(f, f * 2)
        self.enc3 = ConvBlock(f * 2, f * 4)
        self.enc4 = ConvBlock(f * 4, f * 8)
        self.bottleneck = ConvBlock(f * 8, f * 16)

        self.pool = nn.MaxPool2d(2)

        self.dec4 = AttentionUpBlock(f * 16, f * 8, f * 8)
        self.dec3 = AttentionUpBlock(f * 8, f * 4, f * 4)
        self.dec2 = AttentionUpBlock(f * 4, f * 2, f * 2)
        self.dec1 = AttentionUpBlock(f * 2, f, f)
        self.outc = nn.Conv2d(f, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        x4 = self.enc4(self.pool(x3))
        x5 = self.bottleneck(self.pool(x4))

        x = self.dec4(x5, x4)
        x = self.dec3(x, x3)
        x = self.dec2(x, x2)
        x = self.dec1(x, x1)
        return self.outc(x)


In [ ]:
# Loss, evaluation, and training utilities
def dice_bce_loss(logits, targets, bce_weight=0.5, smooth=1.0):
    bce = nn.BCEWithLogitsLoss()(logits, targets)
    probs = torch.sigmoid(logits).view(-1)
    t = targets.view(-1)
    inter = (probs * t).sum()
    dice = 1.0 - (2.0 * inter + smooth) / (probs.sum() + t.sum() + smooth)
    return bce_weight * bce + (1.0 - bce_weight) * dice


def _ensure_metric_keys(m, pred_bin, gt_bin):
    if 'iou' not in m:
        pred_fg = pred_bin.astype(bool)
        gt_fg = gt_bin.astype(bool)
        inter = np.logical_and(pred_fg, gt_fg).sum()
        union = np.logical_or(pred_fg, gt_fg).sum()
        m['iou'] = float(inter / (union + 1e-8))
    if 'hd95' not in m:
        m['hd95'] = float(m.get('hausdorff95', 0.0))
    return m


@torch.no_grad()
def evaluate_seg_model(model, loader):
    model.eval()
    loss_meter = AverageMeter()
    all_metrics = []

    for imgs, masks in loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        logits = model(imgs)
        loss = dice_bce_loss(logits, masks)
        loss_meter.update(loss.item(), imgs.size(0))

        preds = (torch.sigmoid(logits) > THRESHOLD).cpu().numpy().astype(np.uint8)
        gt = masks.cpu().numpy().astype(np.uint8)

        for i in range(preds.shape[0]):
            pred_bin = preds[i].squeeze()
            gt_bin = gt[i].squeeze()
            m = compute_segmentation_metrics(pred_bin, gt_bin)
            m = _ensure_metric_keys(m, pred_bin, gt_bin)
            all_metrics.append(m)

    if not all_metrics:
        return float(loss_meter.avg), {
            'dice': 0.0,
            'iou': 0.0,
            'hd95': float('inf'),
            'hausdorff95': float('inf'),
            'hd95_invalid_count': 0,
            'hd95_invalid_rate': 0.0,
        }

    dice_avg = float(np.mean([m['dice'] for m in all_metrics]))
    iou_avg = float(np.mean([m['iou'] for m in all_metrics]))
    hd_values = np.asarray(
        [m.get('hd95', m.get('hausdorff95', float('inf'))) for m in all_metrics],
        dtype=np.float64,
    )
    finite_hd = hd_values[np.isfinite(hd_values)]
    invalid_count = int((~np.isfinite(hd_values)).sum())
    hd_avg = float(np.mean(finite_hd)) if finite_hd.size else float('inf')

    return float(loss_meter.avg), {
        'dice': dice_avg,
        'iou': iou_avg,
        'hd95': hd_avg,
        'hausdorff95': hd_avg,
        'hd95_invalid_count': invalid_count,
        'hd95_invalid_rate': float(invalid_count / hd_values.size) if hd_values.size else 0.0,
    }


def train_seg_model(model, model_name, train_loader, val_loader):
    safe_name = model_name.lower().replace(' ', '_').replace('-', '')
    save_path = MODEL_DIR / f'best_{safe_name}.pth'

    if save_path.exists() and not FORCE_RETRAIN:
        print(f'Loading existing weights for {model_name}: {save_path}')
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        model = model.to(DEVICE)
        return None, save_path

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )
    early_stop = EarlyStopping(patience=PATIENCE)
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_dice': [],
        'val_iou': [],
        'val_hd95': [],
        'val_hd95_invalid_rate': [],
        'lr': [],
    }
    best_dice = -1.0

    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        ep_start = time.time()
        model.train()
        meter = AverageMeter()

        for imgs, masks in train_loader:
            imgs = imgs.to(DEVICE)
            masks = masks.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = dice_bce_loss(logits, masks)
            loss.backward()
            optimizer.step()
            meter.update(loss.item(), imgs.size(0))

        train_loss = float(meter.avg)
        val_loss, val_metrics = evaluate_seg_model(model, val_loader)
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_metrics.get('dice', 0.0))
        history['val_iou'].append(val_metrics.get('iou', 0.0))
        history['val_hd95'].append(val_metrics.get('hd95', float('inf')))
        history['val_hd95_invalid_rate'].append(val_metrics.get('hd95_invalid_rate', 0.0))
        history['lr'].append(optimizer.param_groups[0]['lr'])

        elapsed = time.time() - ep_start
        eta = elapsed * (EPOCHS - epoch)
        hd95_invalid_count = int(val_metrics.get('hd95_invalid_count', 0))
        hd95_note = f' ({hd95_invalid_count} invalid)' if hd95_invalid_count else ''
        print(
            f'[{model_name}] Epoch {epoch:03d}/{EPOCHS} | '
            f'loss={train_loss:.4f}/{val_loss:.4f} | '
            f'dice={val_metrics.get("dice", 0.0):.4f} | '
            f'iou={val_metrics.get("iou", 0.0):.4f} | '
            f'hd95={val_metrics.get("hd95", float("inf")):.4f}{hd95_note} | '
            f'lr={optimizer.param_groups[0]["lr"]:.2e} | '
            f'{fmt_time(elapsed)}/ep ETA {fmt_time(eta)}'
        )

        if val_metrics.get('dice', 0.0) > best_dice:
            best_dice = float(val_metrics['dice'])
            torch.save(model.state_dict(), save_path)
            print(f'  -> {model_name}: saved best checkpoint (Dice={best_dice:.4f})')

        if early_stop(val_loss):
            print(f'  -> {model_name}: early stopping at epoch {epoch}')
            break

    total = time.time() - start
    print(f'[{model_name}] Training finished in {fmt_time(total)}; best Dice={best_dice:.4f}')

    if save_path.exists():
        model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model = model.to(DEVICE)
    return history, save_path


In [ ]:
# Train and evaluate Attention U-Net only, then compose a 3-model comparison table.
model_name = 'Attention U-Net'
seed_everything(SEED)

attention_model = AttentionUNet(in_channels=1, out_channels=1, base_filters=BASE_FILTERS).to(DEVICE)
attention_params = sum(p.numel() for p in attention_model.parameters())
print(f'Running model: {model_name}')
print(f'Parameters: {attention_params:,}')

attention_history, attention_ckpt = train_seg_model(attention_model, model_name, train_loader, val_loader)
attention_test_loss, attention_test_metrics = evaluate_seg_model(attention_model, test_loader)

attention_record = {
    'model': model_name,
    'checkpoint': str(attention_ckpt),
    'params': int(attention_params),
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'patience': PATIENCE,
    'base_filters': BASE_FILTERS,
    'test_loss': float(attention_test_loss),
    'test_dice': float(attention_test_metrics.get('dice', 0.0)),
    'test_iou': float(attention_test_metrics.get('iou', 0.0)),
    'test_hd95': float(attention_test_metrics.get('hd95', 0.0)),
    'test_hd95_invalid_count': int(attention_test_metrics.get('hd95_invalid_count', 0)),
    'test_hd95_invalid_rate': float(attention_test_metrics.get('hd95_invalid_rate', 0.0)),
}
ATTN_RESULTS_PATH.write_text(json.dumps(attention_record, indent=2))
print('Saved Attention U-Net metrics to:', ATTN_RESULTS_PATH)
print(
    f"Attention U-Net test -> loss={attention_record['test_loss']:.4f} | "
    f"dice={attention_record['test_dice']:.4f} | "
    f"iou={attention_record['test_iou']:.4f} | "
    f"hd95={attention_record['test_hd95']:.4f}"
)


RECOMPUTE_BASELINE_TEST_METRICS = True


def _read_json(path: Path, label: str):
    if not path.exists():
        print(f'Warning: {label} metrics file not found -> {path}')
        return {}
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f'Warning: failed to parse {label} metrics from {path}: {e}')
        return {}


def _to_float(value, default=np.nan):
    try:
        if value is None:
            return float(default)
        return float(value)
    except Exception:
        return float(default)


def _metric(payload: dict, *keys, default=np.nan):
    for key in keys:
        if key in payload:
            return _to_float(payload.get(key), default=default)
    return float(default)


def _first_existing_path(candidates):
    for p in candidates:
        path = Path(p)
        if path.exists():
            return path
    return None


def _evaluate_checkpoint_metrics(
    model_name: str,
    model_factory,
    checkpoint_candidates,
    metrics_output_path: Path,
    existing_payload: dict,
):
    if existing_payload and not RECOMPUTE_BASELINE_TEST_METRICS:
        return existing_payload

    ckpt_path = _first_existing_path(checkpoint_candidates)
    if ckpt_path is None:
        print(f'Warning: no checkpoint found for {model_name}.')
        return existing_payload if existing_payload else {}

    action = 'Re-evaluating' if existing_payload else 'Evaluating'
    print(f'{action} {model_name} from checkpoint: {ckpt_path}')

    try:
        model = model_factory().to(DEVICE)
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        if isinstance(checkpoint, dict):
            state_dict = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))
        else:
            state_dict = checkpoint
        model.load_state_dict(state_dict)
        params = int(sum(p.numel() for p in model.parameters()))
        test_loss, test_metrics = evaluate_seg_model(model, test_loader)

        payload = {
            'model': model_name,
            'checkpoint': str(ckpt_path),
            'params': params,
            'img_size': IMG_SIZE,
            'batch_size': BATCH_SIZE,
            'epochs': EPOCHS,
            'lr': LR,
            'patience': PATIENCE,
            'base_filters': BASE_FILTERS,
            'test_loss': float(test_loss),
            'test_dice': float(test_metrics.get('dice', np.nan)),
            'test_iou': float(test_metrics.get('iou', np.nan)),
            'test_hd95': float(test_metrics.get('hd95', np.nan)),
            'test_hausdorff95': float(test_metrics.get('hausdorff95', test_metrics.get('hd95', np.nan))),
            'test_hd95_invalid_count': int(test_metrics.get('hd95_invalid_count', 0)),
            'test_hd95_invalid_rate': float(test_metrics.get('hd95_invalid_rate', 0.0)),
        }

        metrics_output_path.write_text(json.dumps(payload, indent=2))
        print(f'Saved standardized {model_name} metrics to: {metrics_output_path}')
        return payload
    except Exception as e:
        print(f'Warning: failed to evaluate {model_name} from {ckpt_path}: {e}')
        return existing_payload if existing_payload else {}


unet_existing = _read_json(METRICS_DIR / 'unet_metrics.json', 'U-Net')
resunet_existing = _read_json(METRICS_DIR / 'resunet_comparison.json', 'ResUNet')

unet_metrics = _evaluate_checkpoint_metrics(
    model_name='U-Net',
    model_factory=lambda: UNet(in_channels=1, out_channels=1, base_filters=BASE_FILTERS),
    checkpoint_candidates=[
        MODEL_DIR / 'best_unet_real_mask.pth',
        MODEL_DIR / 'best_unet.pth',
    ],
    metrics_output_path=METRICS_DIR / 'unet_metrics.json',
    existing_payload=unet_existing,
)

resunet_metrics = _evaluate_checkpoint_metrics(
    model_name='ResUNet',
    model_factory=lambda: ResUNet(in_channels=1, out_channels=1, base_filters=BASE_FILTERS),
    checkpoint_candidates=[
        MODEL_DIR / 'best_resunet.pth',
    ],
    metrics_output_path=METRICS_DIR / 'resunet_comparison.json',
    existing_payload=resunet_existing,
)

comparison_rows = [
    {
        'model': unet_metrics.get('model', 'U-Net'),
        'test_dice': _metric(unet_metrics, 'test_dice', 'dice', default=np.nan),
        'test_iou': _metric(unet_metrics, 'test_iou', 'iou', default=np.nan),
        'test_hd95': _metric(unet_metrics, 'test_hd95', 'hd95', 'hausdorff95', 'test_hausdorff95', default=np.nan),
        'test_hd95_invalid_count': _metric(unet_metrics, 'test_hd95_invalid_count', 'hd95_invalid_count', default=0.0),
        'test_hd95_invalid_rate': _metric(unet_metrics, 'test_hd95_invalid_rate', 'hd95_invalid_rate', default=0.0),
        'test_loss': _metric(unet_metrics, 'test_loss', default=np.nan),
        'params': _to_float(unet_metrics.get('params', np.nan), default=np.nan),
        'checkpoint': unet_metrics.get('checkpoint', np.nan),
    },
    {
        'model': resunet_metrics.get('model', 'ResUNet'),
        'test_dice': _metric(resunet_metrics, 'test_dice', 'dice', default=np.nan),
        'test_iou': _metric(resunet_metrics, 'test_iou', 'iou', default=np.nan),
        'test_hd95': _metric(resunet_metrics, 'test_hd95', 'hd95', 'hausdorff95', 'test_hausdorff95', default=np.nan),
        'test_hd95_invalid_count': _metric(resunet_metrics, 'test_hd95_invalid_count', 'hd95_invalid_count', default=0.0),
        'test_hd95_invalid_rate': _metric(resunet_metrics, 'test_hd95_invalid_rate', 'hd95_invalid_rate', default=0.0),
        'test_loss': _metric(resunet_metrics, 'test_loss', default=np.nan),
        'params': _to_float(resunet_metrics.get('params', np.nan), default=np.nan),
        'checkpoint': resunet_metrics.get('checkpoint', np.nan),
    },
    {
        'model': attention_record['model'],
        'test_dice': attention_record['test_dice'],
        'test_iou': attention_record['test_iou'],
        'test_hd95': attention_record['test_hd95'],
        'test_hd95_invalid_count': attention_record['test_hd95_invalid_count'],
        'test_hd95_invalid_rate': attention_record['test_hd95_invalid_rate'],
        'test_loss': attention_record['test_loss'],
        'params': attention_record['params'],
        'checkpoint': attention_record['checkpoint'],
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df[
    [
        'model',
        'test_dice',
        'test_iou',
        'test_hd95',
        'test_hd95_invalid_count',
        'test_hd95_invalid_rate',
        'test_loss',
        'params',
        'checkpoint',
    ]
]

missing_mask = comparison_df[['test_dice', 'test_iou', 'test_hd95']].isna().any(axis=1)
if missing_mask.any():
    missing_models = comparison_df.loc[missing_mask, 'model'].tolist()
    print('Warning: missing metrics for:', ', '.join(missing_models))

comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)

comparison_payload = {
    'data_root': str(data_root),
    'split_sizes': {
        'train': len(train_pairs),
        'val': len(val_pairs),
        'test': len(test_pairs),
    },
    'shared_hyperparameters': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'lr': LR,
        'patience': PATIENCE,
        'base_filters': BASE_FILTERS,
        'weight_decay': WEIGHT_DECAY,
        'threshold': THRESHOLD,
    },
    'unet': unet_metrics,
    'resunet': resunet_metrics,
    'attention_unet': attention_record,
    'comparison_rows': comparison_rows,
}
COMPARISON_JSON_PATH.write_text(json.dumps(comparison_payload, indent=2))

print('\nSaved 3-model comparison CSV to:', COMPARISON_CSV_PATH)
print('Saved 3-model comparison JSON to:', COMPARISON_JSON_PATH)
comparison_df


In [ ]:
# Plot test metric comparison across the 3 models.
plot_df = comparison_df.copy()
metric_specs = [
    ('test_dice', 'Dice (higher is better)', True),
    ('test_iou', 'IoU (higher is better)', True),
    ('test_hd95', 'HD95 (lower is better)', False),
]

colors = ['#4E79A7', '#F28E2B', '#76B7B2']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metric, title, _higher_is_better) in zip(axes, metric_specs):
    vals = pd.to_numeric(plot_df[metric], errors='coerce').to_numpy(dtype=np.float64)
    finite_mask = np.isfinite(vals)
    inf_mask = np.isinf(vals)
    nan_mask = np.isnan(vals)

    if finite_mask.any():
        cap = float(np.max(vals[finite_mask]))
        if cap <= 0:
            cap = 1.0
        cap = cap * 1.1
    else:
        cap = 1.0

    draw_vals = vals.copy()
    draw_vals[inf_mask] = cap
    draw_vals[nan_mask] = 0.0

    bars = ax.bar(plot_df['model'], draw_vals, color=colors)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)

    for idx, (bar, raw) in enumerate(zip(bars, vals)):
        if np.isnan(raw):
            label = 'missing'
            bar.set_alpha(0.35)
            bar.set_hatch('//')
        elif np.isinf(raw):
            label = 'inf'
        else:
            label = f'{raw:.2f}' if metric == 'test_hd95' else f'{raw:.4f}'

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha='center',
            va='bottom',
            fontsize=10,
        )

plt.tight_layout()
plt.savefig(METRIC_PLOT_PATH, dpi=200, bbox_inches='tight')
plt.show()
print('Saved metric plot to:', METRIC_PLOT_PATH)


In [ ]:
# Plot Attention U-Net training curves (if trained in this run).
if attention_history is None or len(attention_history.get('train_loss', [])) == 0:
    print('No fresh Attention U-Net history found (checkpoint was loaded).')
else:
    epochs = range(1, len(attention_history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].plot(epochs, attention_history['train_loss'], label='Train Loss')
    axes[0, 0].plot(epochs, attention_history['val_loss'], '--', label='Val Loss')
    axes[0, 1].plot(epochs, attention_history['val_dice'], label='Val Dice', color='green')
    axes[1, 0].plot(epochs, attention_history['val_iou'], label='Val IoU', color='tab:orange')
    axes[1, 1].plot(epochs, attention_history['val_hd95'], label='Val HD95', color='tab:red')

    axes[0, 0].set_title('Attention U-Net Loss Curves')
    axes[0, 1].set_title('Attention U-Net Validation Dice')
    axes[1, 0].set_title('Attention U-Net Validation IoU')
    axes[1, 1].set_title('Attention U-Net Validation HD95')

    for ax in axes.ravel():
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(CURVE_PLOT_PATH, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved training-curve plot to:', CURVE_PLOT_PATH)


## Output Files

- Attention U-Net metrics: `outputs/metrics/attention_unet_metrics.json`
- 3-model comparison CSV: `outputs/metrics/unet_resunet_attention_comparison.csv`
- 3-model comparison JSON: `outputs/metrics/unet_resunet_attention_results.json`
- Test metric bar plot: `outputs/figures/unet_resunet_attention_metrics.png`
- Attention U-Net training curve plot: `outputs/figures/attention_unet_training_curves.png`
